# SE-LLM-350M — Kaggle Pre-Training Notebook

**Instructions:**
1. Enable GPU: Settings → Accelerator → P100
2. Add your dataset: `se-llm-data` (containing train.bin, val.bin)
3. Add Kaggle Secrets: `WANDB_API_KEY` and `GITHUB_TOKEN`
4. Click **Run All** — training starts or resumes automatically

> Each session runs up to 12 hours then checkpoints. Next session resumes automatically.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')


In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q wandb tokenizers datasets pyyaml rich

In [ ]:
# ── Cell 3: Clone training code from GitHub ───────────────────
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_REPO = 'YOUR_GITHUB_USERNAME/se-llm-350m'  # ← UPDATE THIS

if not os.path.exists('/kaggle/working/se-llm-350m'):
    # Use token for private repo access
    try:
        token = secrets.get_secret('GITHUB_TOKEN')
        clone_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
    except Exception:
        clone_url = f'https://github.com/{GITHUB_REPO}.git'
    
    !git clone {clone_url} /kaggle/working/se-llm-350m
    print('Repo cloned')
else:
    !git -C /kaggle/working/se-llm-350m pull
    print('Repo updated')

%cd /kaggle/working/se-llm-350m
!ls -la

In [ ]:
# ── Cell 4: Link dataset files ────────────────────────────────
import os

os.makedirs('data/processed', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Link train.bin from Kaggle dataset
DATASET_PATH = '/kaggle/input/se-llm-data'  # ← your Kaggle dataset name

for fname in ['train.bin', 'val.bin']:
    src  = f'{DATASET_PATH}/{fname}'
    dest = f'data/processed/{fname}'
    if os.path.exists(src) and not os.path.exists(dest):
        os.symlink(src, dest)
        print(f'Linked: {dest} → {src}')
    elif os.path.exists(dest):
        print(f'Already linked: {dest}')
    else:
        print(f'WARNING: {src} not found — check dataset name')

# Link tokenizer
os.makedirs('tokenizer', exist_ok=True)
tok_src = f'{DATASET_PATH}/tokenizer.json'
tok_dst = 'tokenizer/tokenizer.json'
if os.path.exists(tok_src) and not os.path.exists(tok_dst):
    os.symlink(tok_src, tok_dst)
    print(f'Linked tokenizer')

# Check sizes
for f in ['data/processed/train.bin', 'data/processed/val.bin']:
    if os.path.exists(f):
        size_gb = os.path.getsize(f) / 1e9
        print(f'{f}: {size_gb:.2f} GB')

In [ ]:
# ── Cell 5: Load previous checkpoint (if any) ─────────────────
import os, glob

CHECKPOINT_BACKUP = '/kaggle/input/se-llm-checkpoints'  # Previous session output

if os.path.exists(CHECKPOINT_BACKUP):
    # Copy checkpoint from previous session's output
    ckpt_files = glob.glob(f'{CHECKPOINT_BACKUP}/*.pt')
    for ckpt in ckpt_files:
        dest = f'checkpoints/{os.path.basename(ckpt)}'
        if not os.path.exists(dest):
            import shutil
            shutil.copy(ckpt, dest)
            print(f'Restored checkpoint: {dest}')
    
    # Find latest checkpoint
    latest = os.path.join(CHECKPOINT_BACKUP, 'latest.pt')
    if os.path.exists(latest):
        import shutil
        shutil.copy(latest, 'checkpoints/latest.pt')
        print('Restored latest.pt')
else:
    print('No previous checkpoint found — starting fresh')

# Show what we have
ckpts = glob.glob('checkpoints/*.pt')
print(f'\nCheckpoints found: {len(ckpts)}')
for c in sorted(ckpts):
    print(f'  {c}')

In [ ]:
# ── Cell 6: Login to W&B ──────────────────────────────────────
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('W&B logged in')
except Exception as e:
    print(f'W&B login failed: {e} — training will continue without logging')

In [ ]:
# ── Cell 7: START / RESUME TRAINING ───────────────────────────
# This is the main training cell.
# --resume auto: automatically continues from latest checkpoint
# Change to --resume none to restart from scratch

!python training/train.py \
    --config configs/350m.yaml \
    --resume auto

In [ ]:
# ── Cell 8: Save checkpoints to output ────────────────────────
# Kaggle saves /kaggle/working/output automatically.
# Copy checkpoints there so the next session can restore them.

import shutil, glob, os

output_dir = '/kaggle/working/output'
os.makedirs(output_dir, exist_ok=True)

for ckpt in glob.glob('checkpoints/*.pt'):
    dest = f'{output_dir}/{os.path.basename(ckpt)}'
    shutil.copy(ckpt, dest)
    print(f'Saved: {dest}')

# Show final checkpoint info
latest = 'checkpoints/latest.pt'
if os.path.exists(latest):
    import torch
    ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    print(f'\nLatest checkpoint:')
    print(f'  Step:   {ckpt["step"]:,}')
    print(f'  Tokens: {ckpt["tokens_processed"]/1e9:.3f}B')
    print(f'  Loss:   {ckpt["val_loss"]:.4f}')
    
    pct = 100 * ckpt['tokens_processed'] / 2_500_000_000
    print(f'  Progress: {pct:.1f}% complete')